In [ ]:
"""
PII Dataset Augmentation Script
--------------------------------

This script augments a Named Entity Recognition (NER) dataset by inserting
synthetic email addresses associated with person names in the text.

Main Steps:
1. Load training and test datasets
2. Extract first and last names from existing PERSON entities
3. Generate realistic synthetic emails
4. Insert emails into token sequences
5. Validate dataset integrity
6. Save processed datasets
"""

import json
import random
from copy import deepcopy


# ---------------------------------------------------------------------
# Dataset Paths
# ---------------------------------------------------------------------

TRAIN_PATH = "/kaggle/input/datasets/abdullahshheikh/pii-masking-data/data.json"
TEST_PATH = "/kaggle/input/datasets/abdullahshheikh/pii-masking-data/test_data.json"

OUTPUT_TRAIN = "/kaggle/working/train_processed.json"
OUTPUT_TEST = "/kaggle/working/test_processed.json"


# ---------------------------------------------------------------------
# Load Dataset
# ---------------------------------------------------------------------

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)


# ---------------------------------------------------------------------
# Extract First and Last Names from PERSON Entities
# ---------------------------------------------------------------------

def extract_names(dataset):
    """
    Extract unique first and last names from PERSON entity spans.

    Args:
        dataset (list): Dataset containing tokens and NER tags.

    Returns:
        tuple: (list_of_first_names, list_of_last_names)
    """

    first_names = set()
    last_names = set()

    for sample in dataset:
        tokens = sample["tokens"]
        tags = sample["ner_tags"]

        current_name = []

        for token, tag in zip(tokens, tags):

            if tag == "B-PER":
                current_name = [token]

            elif tag == "I-PER":
                current_name.append(token)

            else:
                if current_name:
                    first_names.add(current_name[0].lower())
                    if len(current_name) > 1:
                        last_names.add(current_name[-1].lower())
                    current_name = []

        # Handle span ending at sequence end
        if current_name:
            first_names.add(current_name[0].lower())
            if len(current_name) > 1:
                last_names.add(current_name[-1].lower())

    return list(first_names), list(last_names)


first_names, last_names = extract_names(train_data)


# ---------------------------------------------------------------------
# Email Generator
# ---------------------------------------------------------------------

domains = ["gmail.com", "yahoo.com", "outlook.com", "hotmail.com"]


def generate_email(first=None, last=None):
    """
    Generate a realistic synthetic email address.

    Args:
        first (str): First name
        last (str): Last name

    Returns:
        str: Generated email address
    """

    first = first if first else random.choice(first_names)
    last = last if last else random.choice(last_names)
    domain = random.choice(domains)

    patterns = [
        f"{first}.{last}@{domain}",
        f"{first}{last}@{domain}",
        f"{first}_{last}@{domain}",
        f"{first}{random.randint(10,99)}@{domain}"
    ]

    return random.choice(patterns).lower()


# ---------------------------------------------------------------------
# Insert Email into Token Sequence
# ---------------------------------------------------------------------

def insert_email_in_sequence(sample, insert_prob=0.3):
    """
    Insert a synthetic email address into a token sequence.

    Emails are placed after a PERSON entity span and labeled
    with the NER tag B-EMAIL.

    Args:
        sample (dict): Input dataset sample
        insert_prob (float): Probability of inserting an email

    Returns:
        dict: Modified sample
    """

    tokens = deepcopy(sample["tokens"])
    tags = deepcopy(sample["ner_tags"])

    # Identify PERSON spans
    per_spans = []
    current_span = []

    for i, tag in enumerate(tags):

        if tag == "B-PER":
            current_span = [i]

        elif tag == "I-PER" and current_span:
            current_span.append(i)

        else:
            if current_span:
                per_spans.append(current_span)
                current_span = []

    if current_span:
        per_spans.append(current_span)

    # Skip insertion based on probability
    if not per_spans or random.random() > insert_prob:
        return sample

    target_span = random.choice(per_spans)

    # 50% chance email matches name in sentence
    if random.random() < 0.5:
        first = tokens[target_span[0]]
        last = tokens[target_span[-1]] if len(target_span) > 1 else None
        email_val = generate_email(first, last)
    else:
        email_val = generate_email()

    insert_at = target_span[-1] + 1

    connector = random.choice(["(", "at", "email:", ""])

    if connector:
        tokens.insert(insert_at, connector)
        tags.insert(insert_at, "O")
        insert_at += 1

    tokens.insert(insert_at, email_val)
    tags.insert(insert_at, "B-EMAIL")

    if connector == "(":
        tokens.insert(insert_at + 1, ")")
        tags.insert(insert_at + 1, "O")

    return {
        "tokens": tokens,
        "ner_tags": tags,
        "lang": sample.get("lang", "en"),
        "sequence": " ".join(tokens)
    }


# ---------------------------------------------------------------------
# Apply Augmentation
# ---------------------------------------------------------------------

train_data_augmented = [
    insert_email_in_sequence(sample, insert_prob=0.4)
    for sample in train_data
]

test_data_clean = test_data


# ---------------------------------------------------------------------
# Dataset Validation
# ---------------------------------------------------------------------

valid_tags = {"O", "B-PER", "I-PER", "B-EMAIL", "I-EMAIL"}


def validate_dataset(dataset):
    """
    Ensure dataset integrity:
    - Tokens and tags length match
    - Only valid NER tags are used
    """

    for sample in dataset:

        if len(sample["tokens"]) != len(sample["ner_tags"]):
            raise ValueError("Token/tag length mismatch detected")

        for tag in sample["ner_tags"]:
            if tag not in valid_tags:
                raise ValueError(f"Invalid tag found: {tag}")


validate_dataset(train_data_augmented)


# ---------------------------------------------------------------------
# Save Processed Dataset
# ---------------------------------------------------------------------

with open(OUTPUT_TRAIN, "w", encoding="utf-8") as f:
    json.dump(train_data_augmented, f, indent=2)

with open(OUTPUT_TEST, "w", encoding="utf-8") as f:
    json.dump(test_data_clean, f, indent=2)

Train samples: 28516
Test samples: 3650
Unique first names: 9830
Unique last names: 12483
Original train size: 28516
Augmented train size: 28516
Dataset validation passed.
Processed datasets saved.

--- ANALYZING AUGMENTED DATA (20 SAMPLES) ---
Sample #1 (Index 12):
  Names Found: ['Joss', 'Whedon']
  Email Added: josswhedon@yahoo.com
  Full Sequence: Joss Whedon at josswhedon@yahoo.com was credited as executive producer throughout the run of the series , and for the first five seasons ( 1997 – 2001 ) he was also the showrunner , supervising the writing and all aspects of production .

Sample #2 (Index 15):
  Names Found: ['Syverson', 'Yagelski']
  Email Added: yagelskiwaltz@hotmail.com
  Full Sequence: While a primary concern has been the relationship between the writing process and natural places , concepts of spatiality also apply to cyberspace and online writing - in MUDs , MOOs , Internet Relay Chat , Instant Messages , and e-mail ( Syverson , 1999 ; Yagelski at yagelskiwaltz@hotm